# Protein Crosslink Topology Benchmark

This notebook operationalizes six dataset directions:

1. single-crosslink information landscapes;
2. cooperative crosslink subsets of arbitrary order;
3. minimum retained fingerprint-generating sets;
4. stable local-lasso decomposition versus the complete graph;
5. exact equal abstract connectivity with different spatial fingerprints; and
6. natural-versus-unique-rewired robustness.

It is a benchmark driver, not evidence that equal Yamada fingerprints imply isotopy. Every published table should retain failures, crossing caps, projection settings, Repulsor topology status, null-ensemble provenance, sampling design, and zero-variance diagnostics.

## 1. Load the cached reference inputs

In [ ]:
from pathlib import Path

from knotted_graph.applications.protein import (
    FingerprintComputer,
    FingerprintSettings,
    ProteinBatchSettings,
    ProteinManifestEntry,
    extract_crosslink_core,
    run_protein_batch,
)
from knotted_graph.inputs import load_crosslinked_protein

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the KnottedGraph checkout.")

PDB_CACHE = PROJECT_ROOT / "pdb-cache"
RESULTS = PROJECT_ROOT / "results" / "protein_topology" / "benchmark_07"
RESULTS.mkdir(parents=True, exist_ok=True)

specs = {
    "1AOC_A": ("1AOC", {"disulfide"}),
    "3ULK_A": ("3ULK", {"metal_coordination"}),
    "5OSQ_A": ("5OSQ", {"disulfide", "metal_coordination"}),
}
proteins = {}
for sample_id, (pdb_id, types) in specs.items():
    proteins[sample_id] = load_crosslinked_protein(
        PDB_CACHE / f"{pdb_id}.pdb",
        pdb_id=pdb_id,
        chain_ids=["A"],
        allowed_crosslink_types=types,
        download=False,
    )
    protein = proteins[sample_id]
    core = extract_crosslink_core(protein.graph)
    print(
        sample_id,
        "links=", len(protein.crosslinks),
        "full=", (protein.graph.number_of_nodes(), protein.graph.number_of_edges()),
        "core=", (core.number_of_nodes(), core.number_of_edges()),
        "issues=", protein.issues,
    )

## 2. Projection-complexity audit

The exact evaluator is exponential in diagram complexity. Setting `max_crossings=0` makes this cell a projection-only audit: it records the chosen crossing count and returns `FingerprintComplexityError` instead of starting an unbounded exact calculation.

In [ ]:
audit_computer = FingerprintComputer(
    RESULTS / "projection_audit_cache",
    settings=FingerprintSettings(
        num_rotation_samples=5,
        max_crossings=0,
        n_jobs=1,
    ),
)
projection_audit = {}
for sample_id, protein in proteins.items():
    record = audit_computer.compute(extract_crosslink_core(protein.graph))
    projection_audit[sample_id] = record
    print(sample_id, record.status, record.crossing_count, record.error_type)

## 3. Manifest and batch configuration

The default cell below is a three-protein engineering profile and only creates configuration objects. Set `RUN_BATCH=True` after choosing a crossing policy and preparing Repulsor. It is not the population profile; the frozen scientific manifests and final commands are in `examples/protein_topology/README.md`. `exact_subsets="auto"` enumerates all subsets only when the retained core crosslink count is at most the configured cap.

In [ ]:
entries = [
    ProteinManifestEntry(
        sample_id=sample_id,
        source=str(PDB_CACHE / f"{pdb_id}.pdb"),
        pdb_id=pdb_id,
        chain_ids=("A",),
        allowed_crosslink_types=tuple(sorted(types)),
    )
    for sample_id, (pdb_id, types) in specs.items()
]
settings = ProteinBatchSettings(
    include_pairs=True,
    exact_subsets="auto",
    max_exact_crosslinks=10,
    null_replicates=20,
    null_seed=2026,
    null_embedding_mode="canonical_low_crossing",
    repulsion_steps=20,
    conditioned_robustness=True,
    conditioned_max_subset_order=3,
    minimum_generator_max_retained_crosslinks=3,
    repulsor_root=str(PROJECT_ROOT / "external" / "Repulsor"),
    allow_repulsor_certificate_only=True,
    resume=True,
)
print(entries)
print(settings)

In [ ]:
RUN_BATCH = False

if RUN_BATCH:
    details = run_protein_batch(
        entries,
        RESULTS / "full_run",
        fingerprint_settings=FingerprintSettings(
            num_rotation_samples=32,
            max_crossings=24,
            n_jobs=-1,
        ),
        batch_settings=settings,
    )
    for detail in details:
        print(detail["summary"])
else:
    print("Batch skipped. Review projection_audit and native Repulsor availability first.")

## 4. Direction 1 — information-carrying edges

Use `edge_impacts.csv`. The primary fields are `changed` (\(X_i\)), crosslink chemistry, endpoints, exact-evaluation status, crossing count, and runtime. Compute \(f_{top}\) only over successful single-edge comparisons and always publish `failed_single_count` beside it.

## 5. Direction 2 — cooperative subsets

Use `conditioned_subset_impacts.csv`. A subset is strictly cooperative only when it removes baseline excess embedding topology and no non-empty proper subset does. Failed fingerprints remain null—not false. `--conditioned-max-subset-order` extends the exact scan from pairs to triples or higher order. The certified 5OSQ run evaluated 78 pairs and 286 triples with zero failures and found 19 strict pairs plus 3 strict triples. The independent 82-protein disulfide pair run evaluated 207/207 unique pairs; six were information carrying but none was strictly cooperative. Together with 85/85 recovered high-complexity pairs, the disulfide-only survey contains 292 exact pairs and zero strict pairs.

## 6. Direction 3 — minimum generating sets

The deletion records are complemented to recover retained sets. `search_minimum_generating_crosslink_sets` searches retained sets in increasing size and reports either a proven minimum or a rigorous lower bound after a bounded negative search. It never labels an unproved bound as a minimum. The complete certified 5OSQ search evaluated all $2^{13}=8192$ states without failure and proved $m_{top}=13$; only the full 13-edge set reproduces the full fingerprint.

## 7. Directions 4 and 5 — local motifs and abstract connectivity

Direction 4 uses a Topoly-backed minimal-surface detector for the complete local disulfide-lasso multiset and requires stability across mesh densities. The 21-protein validation set contains six same-lasso-signature groups with different full-graph fingerprints. This proves that local lasso decomposition does not determine the global invariant; it does not claim a complete joint knot/theta/handcuff taxonomy.

Direction 5 first buckets edge-expanded multigraphs and then verifies exact attributed graph isomorphism. Two exact connectivity classes contain different spatial fingerprints: 4UWW/5B1R/7WVR and 29LI/4JP6/7PLP. A hash match alone is never accepted, and equal Yamada fingerprints never establish isotopy.

## 8. Direction 6 — natural versus randomized robustness

The primary null exactly enumerates eligible non-native perfect matchings of the same intrachain disulfide endpoints. It is exhaustive when the ensemble has at most 20 states and otherwise samples without replacement. `coordinate_preserving` retains the folded coordinates; certificate-checked Repulsor fallback is applied only when a natural or selected null state exceeds the exact crossing cap.

The final declared recovered exact-evaluable cohort contains 114/114 successful natural analyses and 398/398 successful selected unique nulls. Raw $R_1$ is saturated at zero and is not used for inference. The abstract-conditioned natural-minus-matched-null carrying-edge fraction is -0.078484, bootstrap 95% CI [-0.122403, -0.036190], with paired sign-flip $p=0.0002699973$. Every conditioned inference gate is true. The nested 82-protein no-natural-fallback sensitivity analysis gives -0.063919, bootstrap 95% CI [-0.101866, -0.028066], with $p=0.00072999$. This is conditional on the declared recovered exact-evaluable RCSB-query cohort, not the whole PDB.

## 9. Reproducible CLI run

The final population manifest is `examples/protein_topology/population_conditioned_recovered_v1.csv`:

```bash
uv run kg-protein-topology \
  examples/protein_topology/population_conditioned_recovered_v1.csv \
  results/protein_topology/population_conditioned_recovered_v1 \
  --rotation-samples 32 --max-crossings 40 --no-pairs \
  --exact-subsets none --conditioned-robustness \
  --repulsion-steps 100 --repulsion-max-time 10 \
  --repulsion-free-special-vertices \
  --repulsion-decimation-passes 16 \
  --repulsion-max-points-per-edge 32 --repulsion-fallback-only \
  --null-replicates 20 --null-seed 2026 \
  --null-embedding-mode coordinate_preserving \
  --null-sampling-mode unique_disulfide_matchings \
  --null-repulsion-fallback-steps 100 \
  --null-repulsion-fallback-max-time 10 \
  --null-repulsion-fallback-free-special-vertices \
  --null-repulsion-fallback-decimation-passes 16 \
  --null-repulsion-fallback-max-points-per-edge 32 \
  --repulsor-root external/Repulsor \
  --allow-repulsor-certificate-only --no-resume
```

Keep `run_config.json`, coordinate sources, fingerprint cache, analysis JSON, CSV tables, figures, Repulsor certificates, and package/native revisions together. The frozen query and all analytic manifests are under `examples/protein_topology/`.